# Train a latent → future occupancy head (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/worldmodels_ros2/blob/main/notebooks/train_occupancy_head.ipynb)

Trains `world_model_py.occupancy_head.OccupancyHead` so a real **I-JEPA** encoder
can publish `FutureOccupancy` for the Nav2 costmap layer / DWB critic.

1. simulate seeded episodes (camera frames, exact future occupancy and a camera
   visibility mask, `occupancy_sim`),
2. encode every frame into I-JEPA **patch-token maps** on the GPU,
3. train the head (camera-geometry lift + small BEV CNN, torch on the GPU) and
   export it as a **numpy-only `.npz`**,
4. report pooled IoU over visible cells on held-out episodes vs. baselines
   (`prior`, `persist-gt`, and the same head on raw `pixels`).

**Runtime → Change runtime type → T4 GPU** (free tier is enough), then *Run all*.
Expected time on a T4: ~30–45 min (most of it is the head training).

In [ ]:
# --- parameters ---------------------------------------------------------------
REPO = "https://github.com/rsasaki0109/worldmodels_ros2.git"
BRANCH = "main"            # e.g. "claude/occupancy-head" to test unmerged work
ENCODER = "ijepa"          # "ijepa" or "pixels" (GPU-free baseline)
EPISODES = 120             # training episodes (x 80 frames each)
EPOCHS = 30
OUT = "/content/occ_out"

In [ ]:
!nvidia-smi -L || echo "no GPU -- Runtime > Change runtime type > T4 GPU"

In [ ]:
%cd /content
!rm -rf worldmodels_ros2 && git clone -q --depth 1 -b $BRANCH $REPO
%cd /content/worldmodels_ros2
!git log -1 --oneline
# torch / transformers / pillow are preinstalled on Colab; the package itself is pure python
!pip install -q -e world_model_py

In [ ]:
!WM_DEVICE=cuda python world_model_py/test/train_occupancy_head.py \
    --encoder $ENCODER --episodes $EPISODES --epochs $EPOCHS --out $OUT

## Results

`preview.png`: camera | prediction vs. ground truth at t+1, t+4, t+8
(white = hit, green = missed, red = false alarm, dark blue = not visible at t
and not scored; forward is up). IoU is pooled over cells visible at t.

In [ ]:
import json
from IPython.display import Image, display
display(Image(f"{OUT}/preview.png"))
m = json.load(open(f"{OUT}/metrics.json"))
for k, v in m["iou_mean"].items():
    steps = " ".join(f"{x:.2f}" for x in m["iou_per_step"][k])
    print(f"{k:12s} visible IoU {v:.3f} (all cells {m['iou_mean_all_cells'][k]:.3f})   per step {steps}")

## Export

Download the zip (head `.npz` + `metrics.json` + `preview.png`) and paste the
printed IoU table back into the issue / chat. Use the head from ROS 2 with:

```bash
ros2 run world_model_py runtime_node --ros-args \
    -p adapter:=ijepa -p occupancy_head:=/path/to/occupancy_head_ijepa.npz
```

In [ ]:
!cd $OUT && zip -q -r /content/occ_out.zip . && ls -la $OUT
from google.colab import files
files.download("/content/occ_out.zip")
# or keep it in Drive:
# from google.colab import drive; drive.mount("/content/drive")
# !cp /content/occ_out.zip /content/drive/MyDrive/